In [0]:
from pyspark.sql import functions as F

### Read and Write Dimension Stream using Auto Loader (cloudFiles)

In [0]:
def ingest_dim_data(table_name):
    # Direct path to directory containing the CSV file
    raw_data_path = f"/Volumes/ecommerce/source_data/raw/{table_name}/"
    checkpoint_path = f"/Volumes/ecommerce/source_data/raw/_checkpoints/brz_{table_name}"
    schema_location_path = f"/Volumes/ecommerce/source_data/raw/_schemas/brz_{table_name}"

    df_dim_stream = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_location_path)
        .option("header", "true")
        .option("inferSchema", "true")
        .load(raw_data_path)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
    )

    (df_dim_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(f"ecommerce.bronze.brz_{table_name}")
    )

### Execute Dimension Tables

In [0]:

tables = ['brands', 'customers', 'category', 'products', 'date']

for table in tables:
    print(f"Ingesting {table}...")
    ingest_dim_data(table)

print("All done!")

Ingesting brands...
Ingesting customers...
Ingesting category...
Ingesting products...
Ingesting date...
All done!



### Number of Rows and Columns

In [0]:
# tables = spark.sql("SHOW TABLES IN ecommerce.bronze").select("tableName").collect()

# spark.sql(" UNION ALL \n".join([f"SELECT '{row.tableName}' AS table_name, COUNT(*) AS row_count FROM ecommerce.bronze.{row.tableName}" for row in tables])).display()